# D3b — Table extraction: single-call PROMPT arm

A **single-call prompt** (one LLM call per paper, the whole table at once) vs the production
**row_then_columns** two-stage extractor (1 + N calls per paper). Same model as production
(`claude-sonnet-4-6`), **one form at a time**.

- Prompt files: `eval/studies/table_ablation/prompts/<slug>__<key>.md` — edit by hand to tune.
- Output CSV: `eval/sheets/ai sheets/table/<slug>/claude/<key>_prompt_long.csv`.
- **Kernel:** the `topics` env (dspy + litellm). **Scoring is CLI-only** (see Notes).

In [ ]:
# Cell 1 — environment setup (pick the `topics` kernel: dspy + litellm)
import sys, os, asyncio
sys.path.insert(0, '/home/ubuntu/evistream')
sys.path.insert(0, '/home/ubuntu/evistream/backend')
import pandas as pd
try:
    import dspy, litellm  # noqa: F401
except ModuleNotFoundError as e:
    raise SystemExit(f'Missing {e.name} — switch to the topics env kernel.')
from dotenv import load_dotenv
for p in ('/home/ubuntu/evistream/eval/.env', '/home/ubuntu/evistream/backend/.env'):
    if os.path.exists(p):
        load_dotenv(p, override=False)
print('python   :', sys.executable)
print('litellm  :', litellm.__version__)
print('ANTHROPIC_API_KEY set:', bool(os.getenv('ANTHROPIC_API_KEY')))


In [ ]:
# Cell 2 — config (one form at a time) + resolve the form
from eval.studies.table_ablation.forms_registry import FORMS
from eval.studies.table_ablation.run_prompt_arm import _load_papers, _extract_one, _write_csv, MODEL
from eval.studies.table_ablation.transform import (explode_table_results, explode_table_grounded,
                                           grounded_table_header, subform_cols, table_field_name)
from eval.studies.ablation.run_extraction import _write_grounded_csv
from eval.studies.table_ablation.seed_prompts import load_schema_def

# ---- knobs ----
FORM         = 'perio_interventions'   # one of sorted(FORMS) printed below
PAPERS_LIMIT = None                    # None = all papers; an int = smoke test
CONCURRENCY  = 8

print('available forms:', sorted(FORMS))
spec       = FORMS[FORM]
schema_def = load_schema_def(spec)
field      = table_field_name(schema_def)
cols       = subform_cols(schema_def, field)
print(f'\nFORM={FORM}   model={MODEL}')
print(f'  table field : {field}  ({len(cols)} columns)')
print(f'  papers cache: {spec.markdown_dir}')
print(f'  prompt file : {spec.prompt_path}')
print(f'  out CSV     : {spec.out_csv}')
print(f'  production  : {spec.production_csv}  (row_then_columns arm, for CLI scoring)')


In [ ]:
# Cell 3 — the exact system prompt sent to the model (one call/paper) for this form.
# Tune by editing the .md by hand; re-seed from the schema via CLI:
#   python -m eval.studies.table_ablation.run_prompt_arm --form <key> --seed --force
print(spec.prompt_path.read_text())


In [ ]:
# Cell 4 — what's cached so far (all forms) + optional start-from-scratch
START_FROM_SCRATCH = False   # set True to DELETE this form's cached CSV before re-running Cell 5

print('cached prompt CSVs:')
for k in sorted(FORMS):
    p = FORMS[k].out_csv
    if p.exists():
        n = max(0, sum(1 for _ in open(p)) - 1)
        print(f'  {"*" if k == FORM else " "} {k:30} cached  ({n} rows)')
    else:
        print(f'  {"*" if k == FORM else " "} {k:30} -- not yet')

if START_FROM_SCRATCH and spec.out_csv.exists():
    spec.out_csv.unlink()
    print(f'\ndeleted {spec.out_csv}\nCell 5 will re-extract this form from scratch.')
# wipe ALL forms:  for k in FORMS: FORMS[k].out_csv.unlink(missing_ok=True)


In [ ]:
# Cell 5 — extract the single-call PROMPT arm (ONE litellm call/paper, parallel). Skips if cached.
if spec.out_csv.exists():
    print(f'cached -> {spec.out_csv}   (Cell 4: START_FROM_SCRATCH=True to redo)')
    df = pd.read_csv(spec.out_csv)
else:
    litellm.drop_params = True
    system_prompt = spec.prompt_path.read_text()
    papers = _load_papers(spec.markdown_dir, PAPERS_LIMIT)
    print(f'extracting {len(papers)} papers with {MODEL} (concurrency={CONCURRENCY}) ...')
    sem = asyncio.Semaphore(CONCURRENCY)
    pairs = await asyncio.gather(*[_extract_one(system_prompt, p, sem) for p in papers])
    results = {doc_id: {field: rows} for doc_id, rows in pairs}
    rows = explode_table_results(results, papers, field, cols)
    spec.out_csv.parent.mkdir(parents=True, exist_ok=True)
    _write_csv(rows, cols, spec.out_csv)
    _write_grounded_csv(explode_table_grounded(results, papers, field, cols),
                        grounded_table_header(cols), spec.grounded_csv)
    df = pd.read_csv(spec.out_csv)
print('value CSV    ->', spec.out_csv)
print('grounded CSV ->', spec.grounded_csv)


In [ ]:
# Cell 6 — eyeball: one row per study arm/record
df = pd.read_csv(spec.out_csv)
print(f'{len(df)} rows  |  {df["Paper"].nunique()} papers  |  columns: {list(df.columns)}')
print('\nrows per paper:')
print(df['Paper'].value_counts())
df.head(12)


## Notes
- **One form at a time:** set `FORM` in Cell 2 to any of `sorted(FORMS)`, re-run Cells 2 -> 6.
- **Full run:** `PAPERS_LIMIT = None` (default); set an int for a smoke test.
- **Start from scratch:** Cell 4 `START_FROM_SCRATCH = True` deletes this form's CSV; wipe all with `for k in FORMS: FORMS[k].out_csv.unlink(missing_ok=True)`.
- **Tune the prompt:** edit `eval/studies/table_ablation/prompts/<slug>__<key>.md`, or re-seed from the schema: `python -m eval.studies.table_ablation.run_prompt_arm --form <key> --seed --force`.
- **Scoring (CLI-only):** `python -m eval.studies.table_ablation.score_prompt_arm --form <key>` -> prompt vs row_then_columns, writes `eval/outputs/table/<slug>/claude/<key>_metrics.xlsx`.
- **CLI extract:** `python -m eval.studies.table_ablation.run_prompt_arm --form <key> [--papers N]`.